# 03 – Sentence-BERT: Anlam Benzerliği

02 numaralı not defterinde kelime tabanlı yöntemin (TF-IDF) en iyi ayarla bile hedefin (Pearson ≥ 0,70) altında kaldığını gördük. Sebebi belliydi: TF-IDF yalnızca **ortak kelimelere** bakıyor; aynı fikri farklı kelimelerle anlatan öğrenciyi tanıyamıyor.

Bu not defterinde ilk gerçek yapay zekâ modelimizi kullanacağız: **Sentence-BERT**. Bu model cümlenin **anlamına** bakar.

Adımlar:
1. Cümle gömmesi (embedding) nedir? Küçük bir gösterim
2. Kullanacağımız iki model
3. Bütün cevapların anlam benzerliğinin hesaplanması
4. 01'deki "ortak kelimesi yok ama 5/5 almış" cevaplar
5. TF-IDF ile Sentence-BERT'in karşılaştırılması
6. Hata analizi: Sentence-BERT nerede yanılıyor?

### Nasıl çalıştırılır?
1. Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle ve çift tıklayıp Colab'da aç.
2. **GPU'yu aç:** Üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**. GPU (ekran kartı) bu modelleri çok hızlandırır: GPU ile yaklaşık 1 dakika, GPU olmadan 10–15 dakika sürer. GPU açılmazsa da not defteri çalışır, sadece daha yavaş olur.
3. **Çalışma zamanı → Tümünü çalıştır** de, Drive izni isterse **İzin ver**.

## 1. Kurulum

Önceki not defterlerindeki kurulumun aynısı. Ek olarak GPU'nun açık olup olmadığını kontrol ediyoruz.

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

if not (PROJE / "data" / "processed" / "ortak_veri.csv").exists():
    for betik in ["scripts/veri_indir.py", "scripts/ortak_bicim.py"]:
        sonuc = subprocess.run([sys.executable, betik], capture_output=True, text=True)
        print(sonuc.stdout[-1500:])
        if sonuc.returncode != 0:
            print(sonuc.stderr[-3000:])
            raise RuntimeError(f"{betik} başarısız oldu; hata mesajı yukarıda.")

try:
    import sentence_transformers  # Colab'da kurulu gelir
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

import torch
import pandas as pd
CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU açık:", torch.cuda.get_device_name(0) if CIHAZ == "cuda" else "HAYIR (CPU ile daha yavaş çalışacak)")

df = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
print(f"{len(df):,} cevap yüklendi.".replace(",", "."))

## 2. Cümle gömmesi (embedding) nedir?

Sentence-BERT bir cümleyi alır ve onu **anlamını temsil eden bir sayı dizisine** çevirir. Bu diziye **gömme** (embedding) ya da **vektör** denir. Kullanacağımız küçük modelde her cümle 384 sayıyla temsil edilir.

Bu sayıların tek tek bir anlamı yoktur; önemli olan, **anlamca benzer cümlelerin sayı dizilerinin de birbirine yakın olmasıdır**. Model bunu, bir milyardan fazla "anlamca benzer cümle çifti" görerek önceden öğrenmiştir. Biz modeli eğitmiyoruz, **hazır hâliyle** kullanıyoruz.

İki cümlenin benzerliğini yine **kosinüs benzerliği** ile ölçüyoruz; farkı, artık kelime sayılarının değil **anlam vektörlerinin** karşılaştırılması.

Bir referans cevap ve dört farklı cevapla TF-IDF ve Sentence-BERT'i yan yana görelim:

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

kucuk_model = SentenceTransformer("all-MiniLM-L6-v2", device=CIHAZ)

referans = "Photosynthesis is the conversion of light energy into chemical energy."
cevaplar = {
    "Aynı anlam, farklı kelimeler": "Plants turn sunlight into food.",
    "Aynı anlam, ortak kelimeler": "In photosynthesis, plants convert light energy to chemical energy.",
    "Yanlış (enerjiler ters çevrilmiş)": "Photosynthesis is the conversion of chemical energy into light energy.",
    "Alakasız": "A compiler translates source code into machine code.",
}

vektor = kucuk_model.encode(referans)
print(f"Referans cevabın gömmesi: {len(vektor)} sayı. İlk 8 tanesi:")
print(vektor[:8].round(3), "...\n")

tum = [referans] + list(cevaplar.values())
tfidf = cosine_similarity(TfidfVectorizer().fit_transform(tum))[0, 1:]
sbert = cosine_similarity(kucuk_model.encode(tum))[0, 1:]
display(pd.DataFrame({"Cevap": list(cevaplar.values()), "TF-IDF benzerliği": tfidf.round(2),
                      "Sentence-BERT benzerliği": sbert.round(2)}, index=list(cevaplar.keys())))

**Tablo ne anlatıyor?**

| Cevap | TF-IDF | Sentence-BERT | Yorum |
|---|---|---|---|
| Aynı anlam, farklı kelimeler | 0,06 | **0,58** | Sentence-BERT'in güçlü yanı: Ortak kelime neredeyse yok ama "bitkiler güneş ışığını besine çevirir" cümlesinin fotosentezden bahsettiğini anlıyor. |
| Aynı anlam, ortak kelimeler | 0,49 | **0,94** | İki yöntem de benzer buluyor; Sentence-BERT çok daha emin. |
| Yanlış (enerjiler ters çevrilmiş) | 1,00 | **1,00** | **İkisi de kandı!** Kelimeler tamamen aynı, sadece yerleri değişmiş. Cümle yanlış ama iki yöntem de "aynı cümle" diyor. |
| Alakasız | 0,04 | 0,24 | İkisi de düşük buluyor. Sentence-BERT'te benzerlik pek 0'a inmez; bu yüzden benzerliği puana çevirirken ölçekleme gerekecek. |

Yani Sentence-BERT **farklı kelimelerle anlatılan aynı fikri** tanıyabiliyor, ama **aynı kelimelerle söylenen zıt fikri** her zaman ayırt edemiyor. Bu ikinci noktayı aşağıda gerçek öğrenci cevaplarında da göreceğiz.

## 3. Kullanacağımız iki model

| | **MiniLM** (`all-MiniLM-L6-v2`) | **MPNet** (`all-mpnet-base-v2`) |
|---|---|---|
| Boyut | 22 milyon parametre, 6 katman | 110 milyon parametre, 12 katman |
| Gömme uzunluğu | 384 sayı | 768 sayı |
| Hız | Çok hızlı | Yaklaşık 5 kat daha yavaş |
| Doğruluk | İyi | Genelde daha iyi |

> **Parametre:** Modelin eğitim sırasında öğrendiği sayılar. Ne kadar çok parametre, model o kadar çok şey öğrenebilir ama o kadar yavaş çalışır.

İkisi de `sentence-transformers` kütüphanesinin en çok kullanılan hazır modelleri. İkisini de deneyip "büyük model, ek yavaşlığına değiyor mu?" sorusuna cevap arayacağız.

Aşağıdaki hücre bütün cevapları ve referansları iki modelle de vektöre çevirir ve her öğrenci cevabının referansla benzerliğini hesaplar. Sonuçlar Drive'a kaydedilir; not defterini tekrar çalıştırırsan hesaplama baştan yapılmaz.

In [ ]:
import time
import numpy as np

MODELLER = {"MiniLM": "all-MiniLM-L6-v2", "MPNet": "all-mpnet-base-v2"}
BENZERLIK_DOSYASI = SONUC / "03_sbert_benzerlik.csv"

if BENZERLIK_DOSYASI.exists():
    benzerlik = pd.read_csv(BENZERLIK_DOSYASI)
    print("Benzerlikler Drive'dan okundu.")
else:
    # Aynı metni iki kez hesaplamamak için benzersiz metinleri bir kez vektöre çeviriyoruz
    metinler = pd.unique(pd.concat([df["referans_cevap"], df["ogrenci_cevap"]]))
    sira = {m: i for i, m in enumerate(metinler)}
    benzerlik = pd.DataFrame(index=df.index)
    for ad, model_adi in MODELLER.items():
        baslangic = time.time()
        model = SentenceTransformer(model_adi, device=CIHAZ)
        vektorler = model.encode(list(metinler), batch_size=128, normalize_embeddings=True,
                                 show_progress_bar=True)
        ref = vektorler[df["referans_cevap"].map(sira).values]
        cvp = vektorler[df["ogrenci_cevap"].map(sira).values]
        benzerlik[ad] = (ref * cvp).sum(axis=1)   # birim vektörlerde nokta çarpım = kosinüs benzerliği
        print(f"{ad}: {len(metinler):,} metin, {time.time() - baslangic:.0f} saniye".replace(",", "."))
    benzerlik.to_csv(BENZERLIK_DOSYASI, index=False)

df["MiniLM"], df["MPNet"] = benzerlik["MiniLM"].values, benzerlik["MPNet"].values

## 4. 01'deki örnekler: Kelimeler farklı, anlam aynı

01 numaralı not defterinde, referansla **hiç ortak kelimesi olmayan** ama öğretmenden **tam puan** almış cevaplar görmüştük. TF-IDF bu cevaplara sıfıra yakın benzerlik veriyordu. Sentence-BERT ne diyor?

Karşılaştırma kolay olsun diye her benzerliğin yanına, o değerin Mohler'deki bütün cevaplar arasında **kaçıncı yüzdelikte** olduğunu da yazıyoruz (yüzdelik 80 = cevapların %80'inden daha benzer).

In [ ]:
import textwrap
from src.puanlama.ozellikler import TfidfKosinus, kelimeler, ngram_ortusme

mohler = df[df["veri_seti"] == "mohler"].copy()
mohler["TF-IDF"] = TfidfKosinus().ogren(pd.concat([mohler["referans_cevap"], mohler["ogrenci_cevap"]])) \
    .benzerlik(mohler["referans_cevap"], mohler["ogrenci_cevap"])
mohler["ortusme"] = [ngram_ortusme(r, c) for r, c in zip(mohler["referans_cevap"], mohler["ogrenci_cevap"])]
mohler["kelime"] = mohler["ogrenci_cevap"].map(lambda s: len(kelimeler(s)))

def yaz(baslik, metin):
    print(f"{baslik:<16}" + textwrap.fill(metin, 100, subsequent_indent=" " * 16))

tam = mohler[(mohler["puan"] == 1.0) & (mohler["kelime"] >= 6)].nsmallest(3, "ortusme")
for _, satir in tam.iterrows():
    print("=" * 116)
    yaz("Referans:", satir["referans_cevap"])
    yaz("Öğrenci (5/5):", satir["ogrenci_cevap"])
    for yontem in ["TF-IDF", "MiniLM", "MPNet"]:
        yuzdelik = (mohler[yontem] < satir[yontem]).mean() * 100
        print(f"{yontem + ':':<16}benzerlik {satir[yontem]:.2f}  (yüzdelik {yuzdelik:.0f})".replace(".", ","))

**Ne görüyoruz?**

- **1. örnek:** TF-IDF bu tam puanlı cevabı Mohler'deki **en düşük** benzerlikli cevaplar arasına koyuyordu (yüzdelik 0). Sentence-BERT onu **ortalara** taşıdı (MiniLM yüzdelik 29, MPNet yüzdelik 46). "class variables" ile "data members" arasındaki anlam yakınlığını kısmen yakalıyor ama yine de tam puanlı bir cevaba yakışacak kadar yüksek değil.
- **2. örnek:** *"whenevery you dont specifiy your own"* cevabını iki yöntem de neredeyse en alta koyuyor. Bu cevapta hem **yazım hataları** var hem de "your own" ifadesi ancak soruyu bilen biri için anlamlı ("kendi constructor'ını yazmadığında"). Öğretmen soruyu bildiği için anlıyor; bizim modelimiz ise **soruyu görmüyor**, sadece referansla cevabı karşılaştırıyor.
- **3. örnek:** Sentence-BERT cevabı ortanın biraz üstüne koyuyor (yüzdelik 56–63); TF-IDF ise en alta koyuyordu.

Sonuç: Sentence-BERT bu cevaplarda TF-IDF'ten **daha iyi**, ama onları tam puanlı cevapların olması gereken üst sıralara taşıyamıyor.

## 5. TF-IDF ile Sentence-BERT'in karşılaştırılması

Üç yöntemi 02'deki test düzeniyle karşılaştırıyoruz (Mohler'de görülmemiş sorularla 5 katlı çapraz doğrulama; SciEntsBank ve Beetle'da görülmemiş cevaplar ve görülmemiş sorular):

| Yöntem | Ne kullanıyor? |
|---|---|
| **TF-IDF** | 02'deki en iyi ayar: kök bulma + üç özellik (kosinüs, n-gram, uzunluk) |
| **MiniLM** | Sadece MiniLM anlam benzerliği |
| **MPNet** | Sadece MPNet anlam benzerliği |

Benzerlik değeri doğrudan puan değildir (örneğin MPNet'te benzerlik 0,3 ile 0,9 arasında değişebilir). Bu yüzden benzerliği puana çevirmek için yine **doğrusal regresyon** kullanıyoruz: Eğitim verisinden "benzerlik şu kadarsa puan şu kadardır" ilişkisini öğreniyor. Bu işlem Pearson korelasyonunu değiştirmez, sadece puanları doğru aralığa taşıyarak QWK'nın anlamlı hesaplanmasını sağlar.

In [ ]:
import re
from nltk.stem import PorterStemmer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold
from src.puanlama.ozellikler import ozellik_tablosu
from src.puanlama.olcutler import olc

# 02'deki en iyi TF-IDF ayarı: kök bulma
_kok_bulucu, _onbellek, _KELIME = PorterStemmer(), {}, re.compile(r"[a-z0-9]+")
def kokle(metin):
    sonuc = []
    for k in _KELIME.findall(metin.lower().replace("n't", " not")):
        if k not in _onbellek:
            _onbellek[k] = _kok_bulucu.stem(k)
        sonuc.append(_onbellek[k])
    return " ".join(sonuc)

kok_df = df.copy()
kok_df["referans_cevap"] = kok_df["referans_cevap"].map(kokle)
kok_df["ogrenci_cevap"] = kok_df["ogrenci_cevap"].map(kokle)

def tahmin_tfidf(egitim_i, test_i):
    egitim, test = kok_df.loc[egitim_i], kok_df.loc[test_i]
    kosinus = TfidfKosinus().ogren(pd.concat([egitim["referans_cevap"], egitim["ogrenci_cevap"]]))
    oz_e, oz_t = ozellik_tablosu(egitim, kosinus), ozellik_tablosu(test, kosinus)
    model = LinearRegression().fit(oz_e, egitim["puan"])
    return np.clip(model.predict(oz_t), 0, 1)

def tahmin_sbert(sutun):
    def tahmin(egitim_i, test_i):
        model = LinearRegression().fit(df.loc[egitim_i, [sutun]], df.loc[egitim_i, "puan"])
        return np.clip(model.predict(df.loc[test_i, [sutun]]), 0, 1)
    return tahmin

YONTEMLER = {"TF-IDF": tahmin_tfidf, "MiniLM": tahmin_sbert("MiniLM"), "MPNet": tahmin_sbert("MPNet")}

def testler():
    """(test adı, eğitim satırları, test satırları, QWK basamağı) üretir."""
    mohler_i = df.index[df["veri_seti"] == "mohler"]
    for egitim, test in GroupKFold(n_splits=5).split(mohler_i, groups=df.loc[mohler_i, "soru_id"]):
        yield "Mohler", mohler_i[egitim], mohler_i[test], 5
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        egitim = df.index[(df["veri_seti"] == veri_seti) & (df["bolum"] == "train")]
        for bolum, aciklama in [("test_ua", "görülmemiş cevaplar"), ("test_uq", "görülmemiş sorular")]:
            yield f"{ad} – {aciklama}", egitim, df.index[(df["veri_seti"] == veri_seti) & (df["bolum"] == bolum)], 2

tahminler = {y: pd.Series(np.nan, index=df.index) for y in YONTEMLER}
test_listesi = {}
for test_adi, egitim_i, test_i, seviye in testler():
    test_listesi.setdefault(test_adi, (seviye, []))[1].append(test_i)
    for yontem, fonksiyon in YONTEMLER.items():
        tahminler[yontem].loc[test_i] = fonksiyon(egitim_i, test_i)

satirlar = []
for test_adi, (seviye, parcalar) in test_listesi.items():
    test_i = np.concatenate(parcalar)
    for yontem in YONTEMLER:
        satirlar.append({"test": test_adi, "yontem": yontem,
                         **olc(df.loc[test_i, "puan"], tahminler[yontem].loc[test_i], seviye)})
sonuc = pd.DataFrame(satirlar)
sonuc.round(4).to_csv(SONUC / "03_sentence_bert.csv", index=False, encoding="utf-8")

TEST_SIRA = list(test_listesi)
for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(sonuc.pivot(index="test", columns="yontem", values=olcut).loc[TEST_SIRA, list(YONTEMLER)].round(3))

In [ ]:
import matplotlib.pyplot as plt

RENK = {"TF-IDF": "#2a78d6", "MiniLM": "#eb6834", "MPNet": "#1baf7a"}
fig, ax = plt.subplots(figsize=(13, 4.8))
genislik = 0.26
for i, (yontem, renk) in enumerate(RENK.items()):
    degerler = sonuc[sonuc["yontem"] == yontem].set_index("test").loc[TEST_SIRA, "pearson"]
    ax.bar(np.arange(len(TEST_SIRA)) + (i - 1) * genislik, degerler, width=genislik - 0.03, color=renk, label=yontem)
ax.axhline(0.70, color="#52514e", linewidth=1, linestyle="--")
ax.text(len(TEST_SIRA) - 0.5, 0.71, "hedef 0,70", ha="right", va="bottom", fontsize=9, color="#52514e")
ax.set_xticks(np.arange(len(TEST_SIRA)), [t.replace(" – ", "\n") for t in TEST_SIRA])
ax.set_ylabel("Pearson korelasyonu"); ax.set_ylim(0, 0.8)
for kenar in ["top", "right"]:
    ax.spines[kenar].set_visible(False)
ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", ncol=3)
ax.set_title("Kelime tabanlı (TF-IDF) ve anlam tabanlı (Sentence-BERT) puanlama", loc="left", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "03_sentence_bert.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne görüyoruz? Beklenmedik bir sonuç.**

- **Sentence-BERT tek başına TF-IDF'i geçemedi.** Pearson değerleri her testte TF-IDF'te daha yüksek: Mohler'de 0,54'e karşı 0,49 (MiniLM) ve 0,46 (MPNet), SciEntsBank'te (görülmemiş cevaplar) 0,46'ya karşı 0,40, Beetle'da (görülmemiş cevaplar) 0,55'e karşı 0,49.
- **QWK'da tablo biraz farklı:** Beetle'da Sentence-BERT daha iyi (görülmemiş cevaplarda 0,34'e karşı 0,23); diğer testlerde iki yöntem birbirine yakın.
- **Büyük model (MPNet) küçük modelden (MiniLM) daha iyi değil;** çoğu testte biraz daha kötü. Üstelik 5 kat yavaş. Hazır benzerlik kullanacaksak MiniLM yeterli.

**Neden böyle oldu?** Karşılaştırma tam olarak eşit değil:
- TF-IDF yöntemi **üç özellik** kullanıyor ve kelime ağırlıklarını **bu veri setinin kendi eğitim verisinden** öğreniyor.
- Sentence-BERT ise **tek bir sayı** veriyor ve bu sayıyı hesaplarken öğretmen puanlarından hiçbir şey öğrenmiyor; genel amaçlı "bu iki cümle ne kadar benzer?" sorusuna cevap veriyor.

Asıl sebep ise bir sonraki bölümde: Aynı sorunun bütün cevapları **aynı konudan** bahsettiği için, "konu benzerliği" doğru cevapla yanlış cevabı ayırmaya yetmiyor.

Bu sonuç, **hazır bir yapay zekâ modelinin göreve göre eğitilmeden kısa cevap puanlamada yetersiz kalabildiğini** gösteriyor. Raporda dürüstçe yazılması gereken önemli bir bulgu.

## 6. Hata analizi: Sentence-BERT nerede yanılıyor?

Bir modelin nerede başarısız olduğunu bilmek, sonraki adımı belirler. SciEntsBank ve Beetle'da öğretmenlerin **"çelişkili"** (contradictory) olarak etiketlediği cevaplar var: Referansla aynı konudan bahseden ama **tersini söyleyen** cevaplar. Bunlara Sentence-BERT ne kadar benzerlik veriyor?

In [ ]:
semeval = df[df["veri_seti"].isin(["scientsbank", "beetle"])]
etiket_tr = {"correct": "doğru", "partially_correct_incomplete": "kısmen doğru", "contradictory": "çelişkili",
             "irrelevant": "alakasız", "non_domain": "alan dışı"}
ozet = semeval.groupby("orijinal_puan")["MPNet"].agg(["mean", "size"]).rename(index=etiket_tr)
ozet.columns = ["ortalama MPNet benzerliği", "cevap sayısı"]
display(ozet.loc[list(etiket_tr.values())].round(3))

print("\nÇelişkili olduğu hâlde MPNet'in en yüksek benzerlik verdiği 3 cevap:")
for _, satir in semeval[semeval["orijinal_puan"] == "contradictory"].nlargest(3, "MPNet").iterrows():
    print("=" * 116)
    yaz("Soru:", satir["soru"])
    yaz("Referans:", satir["referans_cevap"])
    yaz("Öğrenci:", satir["ogrenci_cevap"])
    print(f"{'MPNet:':<16}{satir['MPNet']:.2f}   Öğretmen: çelişkili (0 puan)".replace(".", ","))

**Ne görüyoruz?**

- Ortalama benzerlikler: doğru 0,63 · kısmen doğru 0,53 · **çelişkili 0,51** · alakasız 0,42 · alan dışı 0,06.
- **Çelişkili cevaplar, kısmen doğru cevaplarla neredeyse aynı benzerliği alıyor.** Model "referansın tersini söylüyor" ile "referansın bir kısmını söylüyor" arasındaki farkı göremiyor.
- En çarpıcı örnek: Referans *"Corn syrup is **denser** than water"*, öğrenci *"Corn syrup is **less dense** than water"*. Tam tersi, ama benzerlik **0,97**!

**Neden?** Sentence-BERT her cümleyi **ayrı ayrı** vektöre çeviriyor, sonra iki vektörü karşılaştırıyor. Vektör cümlenin **konusunu** (mısır şurubu, su, yoğunluk) çok iyi özetliyor ama "daha yoğun / daha az yoğun" gibi **yön bilgisini** kaybediyor.

**Bundan sonra ne yapabiliriz?**
1. **Özellik birleştirme:** TF-IDF'in kelime özellikleriyle Sentence-BERT'in anlam benzerliğini tek modelde birleştirmek. İki yöntem farklı şeyleri yakaladığı için birbirini tamamlayabilir.
2. **İnce ayar (fine-tuning):** İki cümleyi **birlikte okuyan** bir modeli (çapraz kodlayıcı) öğretmen puanlarıyla eğitmek. Böyle bir model "denser" ile "less dense" kelimelerini yan yana gördüğü için zıtlığı yakalayabilir. Hedefe ulaşmak için en güçlü adayımız bu.

## Özet

| Bulgu | Sonucu |
|---|---|
| Sentence-BERT, farklı kelimelerle anlatılan aynı fikri tanıyor (TF-IDF 0,06 → SBERT 0,58) | Anlam benzerliği gerçekten işe yarıyor |
| Ama zıt anlamlı cevaplara da yüksek benzerlik veriyor ("denser" / "less dense": 0,97) | Konu benzerliği, doğruyu yanlıştan ayırmaya yetmiyor |
| Tek başına TF-IDF'i geçemedi (Pearson: Sentence-BERT 0,23–0,49, TF-IDF 0,34–0,55) | Hazır model, göreve göre eğitilmeden yetersiz kalıyor |
| MPNet, MiniLM'den daha iyi değil ve 5 kat yavaş | Hazır benzerlik için MiniLM yeterli |
| Model soruyu görmüyor, yazım hatalarında zorlanıyor | İleride soru metni de modele verilmeli |

Sonuçlar Drive'da `CevapAnaliz/sonuclar/03_sentence_bert.csv`, grafik `CevapAnaliz/sekiller/03_sentence_bert.png` dosyasına kaydedildi. Her cevabın Sentence-BERT benzerliği de `sonuclar/03_sbert_benzerlik.csv` dosyasında; sonraki not defteri bunu tekrar hesaplamadan kullanacak.

**Sonraki not defteri: 04 – Özellik Birleştirme.** TF-IDF özellikleriyle Sentence-BERT benzerliğini tek bir modelde birleştirip birbirlerini tamamlayıp tamamlamadıklarına bakacağız.